# Exercise 2 — Decision trees & random forests

This exercise accompanies **Lecture 2**. You will:

- implement the **Gini impurity** and a **best-split search** from scratch,
- verify that scikit-learn's tree picks *exactly the same split*,
- watch **overfitting** happen as trees grow — and how to stop it,
- benchmark **single trees vs random forests** on two real datasets.

**Datasets:** hand-picked 1-D toy data, `make_moons`, wine, breast cancer —
all bundled with scikit-learn.

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Wickstrom/eugloh-machine-learning-handbook/main?urlpath=lab/tree/notebooks/02/decision_trees_random_forests_exercise.ipynb)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, load_wine, load_breast_cancer
from sklearn.tree import DecisionTreeClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

## Part A — A decision tree by hand

### A.1 — Six points, one feature

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
y = np.array([0, 0, 0, 1, 0, 1])

fig, ax = plt.subplots(figsize=(7, 1.6))
ax.scatter(x, np.zeros_like(x), c=y, cmap="RdYlGn", s=110, edgecolor="k")
ax.set_yticks([])
ax.set_xlabel("$x$")
ax.set_title("Where would you split?")
plt.show()

### A.2 — Gini impurity

$$G(\text{node}) = 1 - \sum_k p_k^2$$

where $p_k$ is the fraction of class-$k$ samples in the node.

In [ ]:
# YOUR CODE HERE
def gini(labels):
    ...

# Sanity checks — these should all pass:
assert gini(np.array([0, 0, 1, 1])) == 0.5, "50/50 is maximally impure"
assert gini(np.array([1, 1, 1])) == 0.0, "pure node has zero impurity"
print("gini() looks correct")

### A.3 — The cost of a split

The split's quality is the **size-weighted average** of the children's Gini:

$$G(\text{split at } t) = \frac{n_L}{n} G(\text{left}) + \frac{n_R}{n} G(\text{right})$$

In [ ]:
# YOUR CODE HERE
def weighted_gini(x, y, t):
    """Weighted Gini of the split x <= t vs x > t."""
    ...

print(f"weighted Gini at t = 3.5: {weighted_gini(x, y, 3.5):.3f}")   # expected ≈ 0.222

### A.4 — Search for the best split

scikit-learn evaluates every candidate threshold (the midpoints between
consecutive sorted feature values). Do the same: loop over the midpoints,
compute the weighted Gini, keep the best.

In [ ]:
# YOUR CODE HERE
def find_best_split(x, y):
    """Return (best_threshold, best_weighted_gini)."""
    ...

best_t, best_s = find_best_split(x, y)
print(f"best split: x <= {best_t}, weighted Gini = {best_s:.3f}")
# Expected: x <= 3.5, score ≈ 0.222

### A.5 — Does scikit-learn agree?

Fit `DecisionTreeClassifier(max_depth=1)` and inspect the root split via
`stump.tree_.threshold[0]` and `stump.tree_.feature[0]` (feature `-2` means
"leaf"; the root here is feature 0).

In [ ]:
stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(x.reshape(-1, 1), y)
print(f"sklearn root: feature {stump.tree_.feature[0]}, threshold {stump.tree_.threshold[0]:.1f}")
# YOUR CODE HERE: does the threshold match your best_t?
...

### A.6 — Your stump vs scikit-learn's

Turn your best split into a predictor: each side predicts its **majority
class**. Compare your predictions with `stump.predict` on the training points.

In [ ]:
# YOUR CODE HERE
def predict_stump(x_new):
    """Predict 0/1 using your best split and the per-side majority classes."""
    ...

# my_pred = predict_stump(x)
# print(np.array_equal(my_pred, stump.predict(x.reshape(-1, 1))))  # expected: True

## Part B — Watching overfitting happen

### B.1 — Noisy moons, a train/test split

In [ ]:
X_m, y_m = make_moons(n_samples=400, noise=0.3, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=0.3, random_state=0, stratify=y_m)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(X_m[:, 0], X_m[:, 1], c=y_m, cmap="RdYlGn", s=15)
ax.set_aspect("equal")
plt.show()

### B.2 — Accuracy vs `max_depth`

Loop over depths 1–20, record **train** and **test** accuracy, and plot both
curves. Where does test accuracy peak? What happens after?

In [ ]:
depths = range(1, 21)
train_acc, test_acc = [], []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=0)
    # YOUR CODE HERE: fit on the training set, append both accuracies
    ...

# YOUR CODE HERE: plot both curves against depths (legend: train / test)
...

### B.3 — Boundaries at depths 1, 3 and 15

Run the cell and watch the boundary go from a single cut → a few cuts →
pure memorisation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, d in zip(axes, [1, 3, 15]):
    clf = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    xx, yy = np.meshgrid(np.linspace(X_m[:, 0].min() - 0.5, X_m[:, 0].max() + 0.5, 300),
                         np.linspace(X_m[:, 1].min() - 0.5, X_m[:, 1].max() + 0.5, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="RdYlGn")
    ax.scatter(X_m[:, 0], X_m[:, 1], c=y_m, cmap="RdYlGn", s=12)
    ax.set_title(f"depth {d} · test acc {clf.score(X_te, y_te):.2f}")
    ax.set_aspect("equal")
plt.show()

### B.4 — An alternative: `min_samples_leaf`

Keep `max_depth=None` (unlimited growth) but require every leaf to hold at
least `min_samples_leaf` samples. Try `[1, 2, 5, 10, 20, 50]` and plot the
test accuracy. Which knob do you prefer — and why?

In [ ]:
# YOUR CODE HERE
...

## Part C — Single tree vs random forest on real data

The **wine** dataset: 178 samples, 13 chemical features, 3 grape varieties.

### C.1 — Load and split

In [ ]:
wine = load_wine()
X_w, y_w = wine.data, wine.target

# YOUR CODE HERE: split 70/30, stratified, random_state=0
...

### C.2 — An unlimited tree

Fit a `DecisionTreeClassifier` **without** any depth limit. Compare train
and test accuracy — how large is the overfitting gap?

In [ ]:
# YOUR CODE HERE
...

### C.3 — The forest strikes back

Compare (5-fold cross-validation with `cross_val_score`):

1. `DecisionTreeClassifier(max_depth=3)`
2. `RandomForestClassifier(n_estimators=200, random_state=0)`

Report mean ± std of the CV accuracy for both models.

In [ ]:
# YOUR CODE HERE
# Expected: the forest wins by a comfortable margin
...

### C.4 — What did the forest learn?

Fit the forest on the training data and plot `feature_importances_` as a
horizontal bar chart (label the bars with `wine.feature_names`). Which
three features matter most?

In [ ]:
# YOUR CODE HERE
...

### C.5 — Where is it wrong?

Show the confusion matrix of the forest on the test set. Which classes get
confused with each other?

In [ ]:
# YOUR CODE HERE: ConfusionMatrixDisplay.from_estimator or from_predictions
...

## Part D — (Bonus) breast cancer: trees vs the linear model

Rerun last week's dataset with this week's model: `RandomForestClassifier`
vs the scaled `LogisticRegression` pipeline (5-fold CV on the full data).
Which family wins here — and does that contradict the wine result?

In [ ]:
# YOUR CODE HERE
...

---

You have now built the core of a decision tree **by hand** and verified it
against scikit-learn. Next: **neural networks**
(`notebooks/03/neural_networks_exercise.ipynb`).